In [1]:
import torch
import pandas as pd
from tqdm import tqdm
from config import LAM_VALUES, TSR_DIR, PT_TSR_DIR, PROMPTS_FILE, MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE
import gc

REPLICA_EXCHANGE = True
LAM_VALUES = LAM_VALUES
INDEX_UNTIL = 6

# ── Load prompts once ─────────────────────────────────────────────────────────
prompts = pd.read_csv(PROMPTS_FILE, usecols=["text"], nrows=INDEX_UNTIL)["text"].tolist()
print(f"Loaded {len(prompts)} prompts")

gc.collect()
torch.cuda.empty_cache()

from diffusers import StableDiffusion3Pipeline

# ── Load model once ───────────────────────────────────────────────────────────
pipe = StableDiffusion3Pipeline.from_pretrained(
	"stabilityai/stable-diffusion-3-medium-diffusers",
	torch_dtype=torch.float16,
	cache_dir=MODEL_CACHE,
)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)
pipe.transformer = torch.compile(pipe.transformer, mode="reduce-overhead", fullgraph=True) # new code

if REPLICA_EXCHANGE:
	BASE_OUTPUT_DIR = PT_TSR_DIR
else:
	BASE_OUTPUT_DIR = TSR_DIR

lam_dirs = {l: BASE_OUTPUT_DIR / f"lam{l:.3f}".replace(".", "p") for l in LAM_VALUES}
for d in lam_dirs.values():
    d.mkdir(parents=True, exist_ok=True)

# ── Sweep ─────────────────────────────────────────────────────────────────────
for idx, prompt in enumerate(prompts):
	if all((BASE_OUTPUT_DIR / f"lam{l:.3f}".replace(".", "p") / f"{idx:05d}.png").exists() for l in LAM_VALUES):
		continue
	
	generator = torch.Generator(device="cuda").manual_seed(SEED + idx)

	for tsr_lam in tqdm(LAM_VALUES, desc=f"idx={idx}"):

		output_dir = lam_dirs[tsr_lam]

		if (output_dir / f"{idx:05d}.png").exists():
			continue

		images = pipe(
			prompt,
			negative_prompt="",
			num_inference_steps=N_INF_STEPS,
			guidance_scale=GUIDANCE_SCALE,
			tsr_lam=tsr_lam,
			tsr_sigma=TSR_SIGMA,
			replica_exchange=REPLICA_EXCHANGE,
			swap_algorithm=SWAP_ALGORITHM,
			generator=generator,
		).images

		out_path = output_dir / f"{idx:05d}.png"
		images[0].save(out_path, icc_profile=None)
		del images
	torch.cuda.empty_cache()

print("\n All k values complete.")

Loaded 6 prompts


Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]


 All k values complete.


In [ ]:
from fid import compute_sweep

compute_sweep(
	lam_values=LAM_VALUES,
	replica_exchanges=[True, False],
	device="cuda",
	target_indices=None,
	index_until=INDEX_UNTIL,
)